# Aggregate results — across-category comparison

Reads every per-cell interval CSV in `results/`, writes
`results/metrics_long.csv` with overall / in-crisis / out-of-crisis
metrics per `(model, wrapper, ticker, regime)` cell, and renders
the cross-method scatter and rolling-coverage figures into
`report/figures/aggregate/`.

In [ ]:
# sys.path bootstrap so 'from src...' works regardless of CWD
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from pathlib import Path

from src.analysis._shared import RESULTS_DIR, CRISIS_WINDOWS, MODEL_CATEGORIES
from src.plotting import compute_cell_metrics_full

ROOT = RESULTS_DIR.parent


## 1. Load every per-cell interval CSV

In [ ]:
all_files = sorted(f for f in RESULTS_DIR.glob("*.csv")
                   if f.name != "metrics_long.csv"
                   and not f.name.startswith("garch_hardcoded_")
                   and not f.name.startswith("tcp_paper_"))
print(f"Loaded {len(all_files)} interval CSVs")
raw = pd.concat([pd.read_csv(f, parse_dates=["date"]) for f in all_files], ignore_index=True)
raw["regime"] = raw["regime"].astype(str)
raw["category"] = raw["model"].map(MODEL_CATEGORIES)
display(raw.groupby(["category", "model", "wrapper"]).size().to_frame("n_cells"))


## 2. Compute per-cell metrics via src.metrics.Metrics (single source of truth)

In [ ]:
rows = []
for (model, wrapper, ticker, regime), grp in raw.groupby(["model", "wrapper", "ticker", "regime"]):
    cfg = CRISIS_WINDOWS[regime]
    widths = grp["u"] - grp["l"]
    grp2 = grp[np.isfinite(widths)].copy()
    if len(grp2) == 0:
        continue
    m = compute_cell_metrics_full(
        grp2, crisis_start=cfg["onset"], crisis_end=cfg["end"], alpha_nominal=0.10
    )
    rows.append({
        "category": MODEL_CATEGORIES.get(model, "unknown"),
        "model": model, "wrapper": wrapper, "ticker": ticker, "regime": regime,
        **m,
    })

metrics_long = pd.DataFrame(rows)
metrics_long.to_csv(RESULTS_DIR / "metrics_long.csv", index=False)
print(f"wrote {RESULTS_DIR / 'metrics_long.csv'}  ({len(metrics_long)} rows)")
display(metrics_long.dtypes.to_frame("dtype").T)


## 3. Three summary tables (sorted by Winkler)

In [ ]:
DISPLAY_COLS_MAIN = ["cov", "cov_high", "cov_low", "cov_gap", "winkler", "w_high", "w_low", "width"]
DISPLAY_COLS_CRISIS = ["cov_crisis", "winkler_crisis", "width_crisis", "t_recovery", "max_drawdown_cov", "recovered"]

def _agg_table(df, cols):
    return (df.groupby(["category", "model", "wrapper"])[cols]
              .mean()
              .sort_values("winkler" if "winkler" in cols else cols[0])
              .round(3))

print("=" * 80)
print("Table 1 — Main results  (averaged across all ticker × regime cells)")
print("  cov: marginal | cov_high/low: vol-stratified | winkler: primary metric")
print("=" * 80)
display(_agg_table(metrics_long, DISPLAY_COLS_MAIN))

print("\n" + "=" * 80)
print("Table 2 — Crisis-window performance")
print("  cov_crisis/winkler_crisis/width_crisis: within-crisis | t_recovery: days to recover")
print("=" * 80)
display(_agg_table(metrics_long, DISPLAY_COLS_CRISIS))

print("\n" + "=" * 80)
print("Table 3 — Per-regime breakdown (Winkler)") 
print("=" * 80)
display(metrics_long.groupby(["category", "model", "wrapper", "regime"])["winkler"]
        .mean().unstack("regime").round(3)
        .sort_values(list(CRISIS_WINDOWS.keys())))


## 4. Paper figures (generated by scripts/build_artifacts.py — display inline here)

In [ ]:
figs_dir = ROOT / "report" / "figures"
for fname in ["winkler_comparison", "cov_scatter", "width_adaptation", "rolling_coverage"]:
    p = figs_dir / f"{fname}.png"
    if p.exists():
        print(fname)
        display(Image(filename=str(p), width=800))
    else:
        print(f"  {fname}.png not found — run: python scripts/build_artifacts.py")


## 5. Explore metrics_long.csv interactively

In [ ]:
# Reload metrics_long.csv for ad-hoc exploration
ml = pd.read_csv(RESULTS_DIR / "metrics_long.csv")
print(f"{len(ml)} rows, columns: {list(ml.columns)}")
display(ml.head(3))
